In [1]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")


Bootstrap complete. Seed=42


# Real Estate Analysis Crew

## What We're Building

A structured four-agent real estate analysis workflow:

| Agent | Purpose |
|-------|---------|
| Market Analyst | Explain macro and local market dynamics |
| Neighborhood Scorer | Quantify location quality and rental demand signals |
| Pricing Agent | Estimate fair value and pricing position vs comps |
| Risk Reviewer | Combine all evidence into investment risk decision |

## Multi-Agent Analysis Flow

1. Market Analyst sets market context.
2. Pricing Agent evaluates valuation and rent potential.
3. Neighborhood Scorer scores livability and demand factors.
4. Risk Reviewer synthesizes all prior outputs into a final recommendation.

This sequence is educational because each stage has a single responsibility and feeds the next stage with clearer inputs.

## Stack
- CrewAI - multi-agent orchestration
- Ollama - local LLM endpoint


In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")


CrewAI is not installed. Install with: pip install crewai
Running in demo mode without CrewAI runtime.


## Step 2 — Property Details

In [3]:
property_listing = """
PROPERTY LISTING:
Address: 1425 Oak Street, Austin, TX 78704
Type: Single-family home
Beds/Baths: 3BR / 2BA
Sq Ft: 1,850
Year Built: 2018
List Price: $485,000
HOA: None
Lot Size: 0.18 acres
Garage: 2-car attached
Features: Updated kitchen, hardwood floors, smart home system,
          energy-efficient windows, covered patio
Neighborhood: South Congress (SoCo) area
School District: Austin ISD (rated 7/10)
Walk Score: 72
Days on Market: 14

SELLER MOTIVATION: Relocating for work. Priced $10K below recent comp.

COMPARABLE SALES (last 6 months):
- 1501 Oak St: 3BR/2BA, 1,900 sqft, sold $498,000 (45 days ago)
- 1380 Oak St: 3BR/2BA, 1,750 sqft, sold $472,000 (90 days ago)
- 1600 Congress Ave: 4BR/3BA, 2,200 sqft, sold $545,000 (30 days ago)
- 1200 Mary St: 3BR/2BA, 1,600 sqft, sold $440,000 (75 days ago)
"""

investment_goal = "Buy-and-hold rental property. Target 7%+ annual return."

print("Property listing loaded")

Property listing loaded


## Step 3 — Create Agents

In [4]:
if CREWAI_AVAILABLE:
    market_analyst = Agent(
        role="Market Analyst",
        goal="Analyze local market conditions and trends relevant to this property",
        backstory=(
            "You are an Austin housing market specialist. You track inventory, days on market,"
            " financing conditions, and rental trends for investment decisions."
        ),
        llm=llm,
        verbose=True,
    )

    pricing_agent = Agent(
        role="Pricing Agent",
        goal="Estimate fair property value using comparable sales and rental potential",
        backstory=(
            "You are a valuation expert who applies comp adjustments and price-per-sqft analysis"
            " to determine whether a listing is fairly priced."
        ),
        llm=llm,
        verbose=True,
    )

    neighborhood_scorer = Agent(
        role="Neighborhood Scorer",
        goal="Score neighborhood quality for both livability and rental demand",
        backstory=(
            "You evaluate neighborhoods using safety, school quality, amenities, transit,"
            " appreciation trajectory, and tenant demand indicators."
        ),
        llm=llm,
        verbose=True,
    )

    risk_reviewer = Agent(
        role="Risk Reviewer",
        goal="Assess investment risks and provide a buy/pass/negotiate recommendation",
        backstory=(
            "You are a conservative real estate advisor focused on downside protection,"
            " cash flow resilience, and realistic return assumptions."
        ),
        llm=llm,
        verbose=True,
    )
else:
    market_analyst = {"role": "Market Analyst"}
    pricing_agent = {"role": "Pricing Agent"}
    neighborhood_scorer = {"role": "Neighborhood Scorer"}
    risk_reviewer = {"role": "Risk Reviewer"}

print("4 agents created: Market Analyst, Neighborhood Scorer, Pricing Agent, Risk Reviewer")


4 agents created: Market Analyst, Neighborhood Scorer, Pricing Agent, Risk Reviewer


## Step 4 — Create Tasks with Callback

In [5]:
if CREWAI_AVAILABLE:
    market_task = Task(
        description=f"""Analyze the local real estate market for this property:

{property_listing}

Cover:
1. Austin market state (buyers vs sellers dynamics)
2. South Congress submarket trends
3. Pricing and inventory momentum
4. Interest rate impact on affordability
5. Rental market context for 3BR homes""",
        expected_output="Market analysis report for Austin and South Congress.",
        agent=market_analyst,
    )

    pricing_task = Task(
        description=f"""Run a pricing analysis using comparable sales:

{property_listing}

Provide:
1. Price-per-sqft comparison
2. Adjustments for key comp differences
3. Fair value range
4. Pricing verdict (below/fair/above market)
5. Likely negotiated purchase range
6. Estimated monthly rent potential""",
        expected_output="Property valuation and pricing verdict.",
        agent=pricing_agent,
        context=[market_task],
    )

    neighborhood_task = Task(
        description=f"""Score this neighborhood for investment suitability:

{property_listing}

Score 1-10 for:
1. Safety
2. Schools
3. Walkability
4. Transit
5. Amenities
6. Appreciation potential
7. Rental demand

Then compute a weighted overall score and explain weighting.""",
        expected_output="Neighborhood scorecard with overall weighted score.",
        agent=neighborhood_scorer,
        context=[market_task],
    )

    risk_task = Task(
        description=f"""Provide final investment recommendation.

Investment goal: {investment_goal}

Using market, pricing, and neighborhood outputs, deliver:
1. Monthly cash flow estimate
2. Cash-on-cash return estimate
3. Cap rate estimate
4. Break-even occupancy estimate
5. Top 5 risks
6. Mitigation actions
7. Final verdict: BUY, PASS, or NEGOTIATE with target price""",
        expected_output="Integrated risk review and recommendation.",
        agent=risk_reviewer,
        context=[market_task, pricing_task, neighborhood_task],
    )
else:
    market_task = {"name": "Market analysis task"}
    pricing_task = {"name": "Pricing task"}
    neighborhood_task = {"name": "Neighborhood scoring task"}
    risk_task = {"name": "Risk review task"}

print("4 tasks created with flow: market -> pricing + neighborhood -> risk review")


4 tasks created with flow: market -> pricing + neighborhood -> risk review


## Step 5 — Run the Crew

In [6]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Final verdict: NEGOTIATE. Strong location and demand, but margin improves at lower entry price."
        self.tasks_output = [
            _DemoTaskOutput("Market", "Stable demand with moderate inventory pressure and resilient rental fundamentals."),
            _DemoTaskOutput("Pricing", "Current listing is near fair value upper bound; negotiation room likely exists."),
            _DemoTaskOutput("Neighborhood", "Weighted score 8.1/10 driven by walkability, amenities, and rental demand."),
            _DemoTaskOutput("Risk", "Main risks: rate volatility and maintenance variance; mitigations proposed."),
        ]


if CREWAI_AVAILABLE:
    realestate_crew = Crew(
        agents=[market_analyst, pricing_agent, neighborhood_scorer, risk_reviewer],
        tasks=[market_task, pricing_task, neighborhood_task, risk_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Real estate crew assembled.")
    result = realestate_crew.kickoff()
    print("\nAnalysis complete.")
else:
    print("Demo mode: showing structured real estate analysis flow without live CrewAI kickoff.")
    result = _DemoResult()


Demo mode: showing structured real estate analysis flow without live CrewAI kickoff.


In [7]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


print("FINAL INVESTMENT RECOMMENDATION")
print("=" * 60)
print(result.raw)

preview("Market Analysis", result.tasks_output[0].raw)
preview("Pricing Analysis", result.tasks_output[1].raw)
preview("Neighborhood Score", result.tasks_output[2].raw)
preview("Risk Review", result.tasks_output[3].raw)


FINAL INVESTMENT RECOMMENDATION
Final verdict: NEGOTIATE. Strong location and demand, but margin improves at lower entry price.

Market Analysis
Stable demand with moderate inventory pressure and resilient rental fundamentals.

Pricing Analysis
Current listing is near fair value upper bound; negotiation room likely exists.

Neighborhood Score
Weighted score 8.1/10 driven by walkability, amenities, and rental demand.

Risk Review
Main risks: rate volatility and maintenance variance; mitigations proposed.


## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Structured specialization | Each agent owns one analysis layer |
| Sequential evidence flow | Risk review only runs after market, pricing, and neighborhood outputs |
| Educational framing | Each section explains what decision the agent is responsible for |
| Decision transparency | Final verdict includes assumptions, risks, and mitigation actions |

## Extensions

- Integrate real listing and comp APIs for live data
- Add uncertainty ranges for rent and expense assumptions
- Export verdict as structured JSON for downstream dashboards
- Add portfolio-level comparison task across multiple properties
